# Smoke test
### Mostly generated by Anthropic Claude

In [1]:
import torch, torch.nn as nn

def block(i, o):
    return nn.Sequential(nn.Conv3d(i, o, 3, padding=1), nn.InstanceNorm3d(o), nn.ReLU(inplace=True),
                         nn.Conv3d(o, o, 3, padding=1), nn.InstanceNorm3d(o), nn.ReLU(inplace=True))

class TinyUNet3D(nn.Module):
    def __init__(self, in_ch=1, n_cls=6, f=16):
        super().__init__()
        self.e1, self.e2, self.e3 = block(in_ch, f), block(f, 2*f), block(2*f, 4*f)
        self.pool = nn.MaxPool3d(2)
        self.up2 = nn.ConvTranspose3d(4*f, 2*f, 2, stride=2); self.d2 = block(4*f, 2*f)
        self.up1 = nn.ConvTranspose3d(2*f, f, 2, stride=2);   self.d1 = block(2*f, f)
        self.out = nn.Conv3d(f, n_cls, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        d2 = self.d2(torch.cat([self.up2(e3), e2], 1))
        d1 = self.d1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)

for shape in [(64, 64, 32), (128, 128, 64)]:   # placeholder sizes
    torch.cuda.reset_peak_memory_stats()
    m = TinyUNet3D().cuda()
    x = torch.randn(1, 1, *shape, device="cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        loss = m(x).float().mean()
    loss.backward()
    print(shape, f"peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB")
    del m, x, loss; torch.cuda.empty_cache()

AttributeError: module 'torch._C' has no attribute '_cuda_resetPeakMemoryStats'